# Model Inference
## Heart Disease Prediction

This notebook demonstrates how to use the trained model for making predictions on new data.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import pickle
import sys
import os
from pathlib import Path

# Add project root to path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('')))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from src.data.preprocess import HeartDiseasePreprocessor

print("Libraries imported successfully!")


## 1. Load Model and Preprocessor


In [ ]:
# Load model and preprocessor
model_path = "../models/best_model.pkl"
preprocessor_path = "../models/preprocessor.pkl"

print("Loading model and preprocessor...")
with open(model_path, 'rb') as f:
    model = pickle.load(f)

preprocessor = HeartDiseasePreprocessor.load(preprocessor_path)

print(f"Model loaded from: {model_path}")
print(f"Preprocessor loaded from: {preprocessor_path}")
print(f"Model type: {type(model).__name__}")


## 2. Single Prediction Example


In [ ]:
# Example patient data
patient_data = {
    'age': 63,
    'sex': 1,
    'cp': 3,
    'trestbps': 145,
    'chol': 233,
    'fbs': 1,
    'restecg': 0,
    'thalach': 150,
    'exang': 0,
    'oldpeak': 2.3,
    'slope': 0,
    'ca': 0,
    'thal': 1
}

# Convert to DataFrame
patient_df = pd.DataFrame([patient_data])
print("Input data:")
print(patient_df)

# Preprocess
patient_processed = preprocessor.transform(patient_df)

# Predict
prediction = model.predict(patient_processed)[0]
probability = model.predict_proba(patient_processed)[0]

print(f"\nPrediction: {prediction} ({'Disease' if prediction == 1 else 'No Disease'})")
print(f"Probability: {probability[1]:.4f} (Disease), {probability[0]:.4f} (No Disease)")


## 3. Batch Prediction


In [ ]:
# Multiple patients
patients_data = [
    {'age': 63, 'sex': 1, 'cp': 3, 'trestbps': 145, 'chol': 233, 'fbs': 1, 
     'restecg': 0, 'thalach': 150, 'exang': 0, 'oldpeak': 2.3, 'slope': 0, 'ca': 0, 'thal': 1},
    {'age': 37, 'sex': 1, 'cp': 2, 'trestbps': 130, 'chol': 250, 'fbs': 0, 
     'restecg': 1, 'thalach': 187, 'exang': 0, 'oldpeak': 3.5, 'slope': 0, 'ca': 0, 'thal': 2},
    {'age': 41, 'sex': 0, 'cp': 1, 'trestbps': 130, 'chol': 204, 'fbs': 0, 
     'restecg': 0, 'thalach': 172, 'exang': 0, 'oldpeak': 1.4, 'slope': 2, 'ca': 0, 'thal': 2}
]

patients_df = pd.DataFrame(patients_data)
print("Input data for batch prediction:")
print(patients_df)

# Preprocess
patients_processed = preprocessor.transform(patients_df)

# Predict
predictions = model.predict(patients_processed)
probabilities = model.predict_proba(patients_processed)

# Create results DataFrame
results_df = patients_df.copy()
results_df['prediction'] = predictions
results_df['probability_disease'] = probabilities[:, 1]
results_df['prediction_label'] = results_df['prediction'].map({0: 'No Disease', 1: 'Disease'})

print("\nPredictions:")
print(results_df[['age', 'sex', 'cp', 'prediction', 'probability_disease', 'prediction_label']])


## 4. Test on Test Set


In [ ]:
# Load test data (if available)
from src.data.preprocess import load_and_clean_data
from sklearn.model_selection import train_test_split

# Load full dataset
X, y = load_and_clean_data("../data/raw/heart_disease.csv")
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocess test data
X_test_processed = preprocessor.transform(X_test)

# Predictions
test_predictions = model.predict(X_test_processed)
test_probabilities = model.predict_proba(X_test_processed)[:, 1]

# Calculate accuracy
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

test_accuracy = accuracy_score(y_test, test_predictions)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))
print(f"\nClassification Report:")
print(classification_report(y_test, test_predictions))


## Summary

Inference complete! The model can be used to make predictions on new patient data. The model is also deployed as a FastAPI service for production use.
